<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/07%20-%20Validade%20e%20Inferencia%20Logica%20na%20Seguranca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 07 - Notebook: Validade de Argumentos e Inferência Lógica na Segurança do AGV

Este notebook implementa o **Provador Dedutivo Formal** exigido como entregável da Aula 07. Ele utiliza tabelas-verdade exaustivas ($2^n$ estados) para validar protocolos de segurança, demonstrar provas por refutação e identificar falácias lógicas que poderiam causar acidentes com o AGV.

---

### Célula 1 (Texto / Markdown)
```markdown
## 1. Motor do Provador Dedutivo Formal

Implementação da classe principal que gera as combinações booleanas, avalia premissas lógicas e determina se um argumento é válido (Tautologia) ou se constitui uma falácia.

In [4]:
import itertools
import pandas as pd
from typing import Callable, List, Dict

class ProvadorDedutivoFormal:
    def __init__(self, variaveis: List[str]):
        """Inicializa o provador com as variáveis proposicionais do sistema."""
        self.variaveis = variaveis
        # Gera todas as 2^n combinações possíveis de Verdadeiro/Falso
        self.espaco_estados = list(itertools.product([False, True], repeat=len(variaveis)))

    def avaliar_argumento(self, premissas: List[Callable], conclusao: Callable) -> Dict:
        """Avalia um argumento usando Tabela-Verdade Exaustiva e Refutação."""
        linhas = []
        argumento_valido = True
        refutacao_insatisfativel = True

        for estado in self.espaco_estados:
            # Mapeia os valores atuais para as variáveis correspondentes
            ctx = dict(zip(self.variaveis, estado))

            # Avalia todas as premissas
            vals_premissas = [p(**ctx) for p in premissas]
            todas_premissas_v = all(vals_premissas)

            # Avalia a conclusão
            val_conclusao = conclusao(**ctx)

            # Teste 1: Validade Semântica (Se premissas V, conclusão DEVE ser V)
            if todas_premissas_v and not val_conclusao:
                argumento_valido = False

            # Teste 2: Refutação (Premissas V E Conclusão Falsa deve ser impossível)
            # {P1, ..., Pk, ~C} |= F
            val_refutacao = todas_premissas_v and (not val_conclusao)
            if val_refutacao:
                refutacao_insatisfativel = False # Encontrou um caso possível, logo falhou na refutação

            # Monta a linha para visualização
            linha = ctx.copy()
            linha["Premissas_V"] = todas_premissas_v
            linha["Conclusão"] = val_conclusao
            linha["Refutação (P ∧ ~C)"] = val_refutacao
            linhas.append(linha)

        df = pd.DataFrame(linhas)

        return {
            "valido": argumento_valido,
            "refutado_com_sucesso": refutacao_insatisfativel,
            "tabela": df
        }

print("Módulo ProvadorDedutivoFormal carregado com sucesso.")

Módulo ProvadorDedutivoFormal carregado com sucesso.


## 2. Testes Formais: Regras Canônicas de Inferência no AGV

Vamos validar duas regras de segurança críticas implementadas no firmware do AGV: o **Modus Ponens** (trip por obstáculo) e o **Modus Tollens** (diagnóstico de falha de energia).

In [5]:
# Instanciando o provador para as variáveis:
# d1: Obstáculo detectado pelo LiDAR
# Trip: Desligamento do Motor M301
provador_mp = ProvadorDedutivoFormal(["d1", "Trip"])

print("--- Teste 1: MODUS PONENS (Trip de Segurança) ---")
# Premissa 1 (Regra): Se obstáculo (d1), então para o motor (Trip) -> d1 => Trip
P1_mp = lambda d1, Trip: (not d1) or Trip  # Equivalência de Implicação Material
# Premissa 2 (Sensor): Obstáculo detectado -> d1
P2_mp = lambda d1, Trip: d1
# Conclusão (Ação): Motor desligado -> Trip
C_mp = lambda d1, Trip: Trip

resultado_mp = provador_mp.avaliar_argumento([P1_mp, P2_mp], C_mp)

print(f"Argumento é VÁLIDO? {resultado_mp['valido']}")
print(f"Prova por Refutação (Insatisfatível)? {resultado_mp['refutado_com_sucesso']}")
display(resultado_mp['tabela'])


print("\n--- Teste 2: MODUS TOLLENS (Diagnóstico do Encoder) ---")
# P: Ponte H Energizada, Q: Encoder girando
provador_mt = ProvadorDedutivoFormal(["P", "Q"])
# Premissa 1: Se Ponte H energizada (P), então Encoder gira (Q) -> P => Q
P1_mt = lambda P, Q: (not P) or Q
# Premissa 2: Encoder não gira -> ~Q
P2_mt = lambda P, Q: not Q
# Conclusão: Ponte H não está energizada -> ~P
C_mt = lambda P, Q: not P

resultado_mt = provador_mt.avaliar_argumento([P1_mt, P2_mt], C_mt)

print(f"Argumento é VÁLIDO? {resultado_mt['valido']}")
display(resultado_mt['tabela'])

--- Teste 1: MODUS PONENS (Trip de Segurança) ---
Argumento é VÁLIDO? True
Prova por Refutação (Insatisfatível)? True


,d1,Trip,Premissas_V,Conclusão,Refutação (P ∧ ~C)
0,False,False,False,False,False
1,False,True,False,True,False
2,True,False,False,False,False
3,True,True,True,True,False



--- Teste 2: MODUS TOLLENS (Diagnóstico do Encoder) ---
Argumento é VÁLIDO? True


,P,Q,Premissas_V,Conclusão,Refutação (P ∧ ~C)
0,False,False,True,True,False
1,False,True,False,True,False
2,True,False,False,False,False
3,True,True,False,False,False


## 3. Detecção de Falácias Formais Industriais

Simulação de um erro comum na programação de CLPs, onde o engenheiro inverte a lógica de causa e consequência, criando uma **Afirmação do Consequente**.

In [6]:
print("--- Teste 3: FALÁCIA - Afirmação do Consequente ---")
# Contexto: Tentativa de diagnosticar se há obstáculo (d1) só porque o AGV está parado (Trip)

# Premissa 1: Se há obstáculo, o robô para. (d1 => Trip)
P1_falacia = lambda d1, Trip: (not d1) or Trip
# Premissa 2: O robô está parado. (Trip)
P2_falacia = lambda d1, Trip: Trip
# Conclusão (Inválida): Logo, há um obstáculo. (d1)
C_falacia = lambda d1, Trip: d1

resultado_falacia = provador_mp.avaliar_argumento([P1_falacia, P2_falacia], C_falacia)

print(f"Argumento é VÁLIDO? {resultado_falacia['valido']}")
print("CUIDADO: Falha de lógica detectada! O AGV pode estar parado por bateria baixa (b1) ou parada manual (s1).")

# Filtrando na tabela exatamente a linha que quebra a lógica (O Contraexemplo)
tabela = resultado_falacia['tabela']
contraexemplo = tabela[(tabela['Premissas_V'] == True) & (tabela['Conclusão'] == False)]

print("\nContraexemplo (Caso que gera Acidente/Erro de Diagnóstico):")
display(contraexemplo)

--- Teste 3: FALÁCIA - Afirmação do Consequente ---
Argumento é VÁLIDO? False
CUIDADO: Falha de lógica detectada! O AGV pode estar parado por bateria baixa (b1) ou parada manual (s1).

Contraexemplo (Caso que gera Acidente/Erro de Diagnóstico):


,d1,Trip,Premissas_V,Conclusão,Refutação (P ∧ ~C)
1,False,True,True,False,True
